# 01g · 골턴-왓슨 분기 과정: pgf의 고정점과 소멸확률 (Galton-Watson Branching Process: pgf Fixed Points and Extinction)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.10 (무한 상태 사슬의 첫 예: 분기 과정, $m\le1$ ⇒ 소멸) · Lawler 2e §2.4 (pgf 합성 $\varphi_{n+1}=\varphi(\varphi_n)$, 소멸확률 정리 증명) · Ross 12e §4.7 ($EX_n=\mu^n$, $\operatorname{Var}X_n$, $\pi_0=\sum_jP_j\pi_0^j$) · G&S 4e §5.4 (소멸확률 = 최소 비음 근, 임계 경우) · Karlin & Taylor 2e Ch. 8 (임계 생존확률 $\sim2/(\sigma^2n)$) |
| 선수 노트북 | 01f, 00c (01e: Recall Q4의 결합 부등식) |
| 예상 소요 | 90분 |
| 상태 | draft |

지금까지의 마르코프 사슬은 상태가 유한했다. 이 노트북은 개체 수가 한없이 자랄 수 있는 **첫 무한 상태 사슬** — 골턴-왓슨 분기 과정(Galton-Watson branching process) — 을 다룬다. 질문은 01c의 도박꾼 파산과 같은 "흡수 확률"이지만, 상태가 무한이라 선형계 $B=NR$을 풀 수 없다. 대신 00c의 **확률생성함수(pgf)** 가 주인공이 된다: 소멸확률 $q$는 자손 pgf $\varphi$의 **가장 작은 고정점**이고, 세대 $n$까지의 소멸확률은 $\varphi$를 0에서 $n$번 합성한 $\varphi_n(0)$이다. 평균 자손 수 $m$이 1을 넘느냐가 운명을 가르되(상전이), "평균이 크다"와 "살아남는다"가 전혀 다른 말임을 시뮬레이션으로 확인한다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson
from scipy.optimize import brentq
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import galton_watson, extinction_probability
from spkit.plots import plot_paths

SEED, rng = rng_for("01g")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기 (주 예제·푸아송·임계 시뮬레이션)
N_CURVE = scale(4_000)    # q(λ) 곡선의 MC 점 하나당 경로 수
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_CURVE={N_CURVE}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (01f) 세부균형이 정상분포를 함의하는 한 줄 증명은?

<details><summary>정답</summary>

$\sum_x\pi(x)p(x,y)=\sum_x\pi(y)p(y,x)=\pi(y)\sum_xp(y,x)=\pi(y)$ (행합 1).

</details>

**Q2.** (01f) Metropolis-Hastings 수락확률의 일반식과, 제안이 대칭일 때의 형태는?

<details><summary>정답</summary>

$\alpha(x,y)=\min\big(1,\frac{\pi(y)q(y,x)}{\pi(x)q(x,y)}\big)$; 대칭 제안 $q(x,y)=q(y,x)$이면 $\min(1,\pi(y)/\pi(x))$.

</details>

**Q3.** (01f) 적분 자기상관 시간 $\tau$의 정의와 길이 $n$ 사슬의 유효 표본 크기는?

<details><summary>정답</summary>

$\tau=1+2\sum_{k\ge1}\rho_k$; $n_{\rm eff}=n/\tau$, 즉 $\operatorname{Var}(\bar f_n)\approx\sigma_f^2\tau/n$.

</details>

**Q4.** (01e) 결합(coupling) 부등식은 무엇이고 수렴 정리에서 어떤 역할을 하는가?

<details><summary>정답</summary>

$\|P^n(x,\cdot)-\pi\|_{TV}\le P(T_c>n)$ ($T_c$는 두 사슬이 만나는 시각). 두 사슬이 확률 1로 만나면($T_c<\infty$ a.s.) 우변 $\to0$이므로 TV 거리 $\to0$.

</details>

**+1.** (00c) 랜덤 합 $S=\xi_1+\dots+\xi_N$ ($N$과 $\xi_i$ 독립, $\xi_i$ i.i.d.)의 pgf는? 그 평균과 분산은?

<details><summary>정답</summary>

$G_S(s)=G_N(\varphi(s))$ ($\varphi$는 $\xi$의 pgf). $ES=EN\cdot E\xi$, $\operatorname{Var}S=EN\cdot\operatorname{Var}\xi+\operatorname{Var}N\cdot(E\xi)^2$. 오늘 정리 1의 증명이 이 세 식의 반복이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **골턴-왓슨 과정 (Galton-Watson process)**: $Z_0=1$이고
  $$Z_{n+1}=\sum_{i=1}^{Z_n}\xi_{n,i},$$
  여기서 $\xi_{n,i}$는 i.i.d. **자손 수**(offspring)이고 분포는 $p_k=P(\xi=k)$, $k=0,1,2,\dots$ ($Z_n=0$이면 빈 합 $=0$). $(Z_n)$은 $\mathbb N=\{0,1,2,\dots\}$ 위의 **마르코프 사슬**이다: $p(i,j)=P(\xi_1+\dots+\xi_i=j)$. 상태 0은 **흡수**($p(0,0)=1$)이고, $p_0>0$이면 $i\ge1$인 모든 상태는 **일시**(transient)다 — $p(i,0)=p_0^i>0$이므로 매 방문마다 흡수될 양의 확률이 있어 무한히 재방문할 수 없다.
- **자손 pgf** $\varphi(s)=E[s^\xi]=\sum_kp_ks^k$, $s\in[0,1]$. $\varphi(0)=p_0$, $\varphi(1)=1$, $\varphi'(1)=m=E\xi$, $\varphi''(1)=E[\xi(\xi-1)]$. 계수가 모두 $\ge0$이므로 $\varphi$는 $[0,1]$에서 **증가·볼록**이고, $p_0+p_1<1$(어떤 $k\ge2$에 $p_k>0$)이면 $\varphi''>0$, 즉 **강볼록**이다.
- **세대 pgf** $\varphi_n(s)=E[s^{Z_n}]$; $\varphi_0(s)=s$, $\varphi_1=\varphi$. 특히 $\varphi_n(0)=P(Z_n=0)$ = "$n$세대 안에 소멸할 확률".
- **소멸 (extinction)**: 사건 $\{\exists n:Z_n=0\}=\bigcup_n\{Z_n=0\}$. 0이 흡수라 $\{Z_n=0\}\subset\{Z_{n+1}=0\}$ — **증가 사건열**이다. **소멸확률** $q=P(\text{소멸})=\lim_n P(Z_n=0)=\lim_n\varphi_n(0)$ (확률의 연속성).
- **분류**: 평균 자손 수 $m$에 따라 **아임계**(subcritical) $m<1$, **임계**(critical) $m=1$, **초임계**(supercritical) $m>1$. $\sigma^2=\operatorname{Var}\xi$.
- **마팅게일 예고**: $W_n=Z_n/m^n$은 $E[W_{n+1}\mid Z_0,\dots,Z_n]=W_n$을 만족한다 (05a/05d에서 마팅게일로 다룬다).

### 2.2 정리 1 — pgf 합성과 평균·분산 (Lawler 2e §2.4; Ross 12e §4.7)

> $\varphi_{n+1}(s)=\varphi_n(\varphi(s))=\varphi(\varphi_n(s))$, 따라서 $\varphi_n=\varphi^{\circ n}$ ($n$중 합성). $EZ_n=m^n$이고
> $$\operatorname{Var}Z_n=\sigma^2m^{n-1}\frac{m^n-1}{m-1}\ (m\ne1),\qquad \operatorname{Var}Z_n=n\sigma^2\ (m=1).$$

**가정이 왜 필요한가**
- **자손 수의 i.i.d.와 세대 간 독립**: $Z_n=k$ 조건 아래 $Z_{n+1}$이 i.i.d. $k$개의 합이어야 $E[s^{Z_{n+1}}\mid Z_n=k]=\varphi(s)^k$. 형제끼리 자손 수가 상관되거나 세대마다 분포가 바뀌면(환경 변동) 합성 공식이 깨진다.
- $m<\infty$: 평균 공식이 의미를 가지려면. 분산 공식은 $\sigma^2<\infty$가 필요하다. pgf 합성 자체는 무한 평균에서도 성립한다.

<details><summary>증명</summary>

**합성.** $Z_n=k$가 주어지면 $Z_{n+1}=\xi_{n,1}+\dots+\xi_{n,k}$는 i.i.d. $k$개의 합이므로(00c 랜덤 합) $E[s^{Z_{n+1}}\mid Z_n=k]=\varphi(s)^k$. 전확률로
$$\varphi_{n+1}(s)=E\big[E[s^{Z_{n+1}}\mid Z_n]\big]=E\big[\varphi(s)^{Z_n}\big]=\varphi_n(\varphi(s)).$$
$\varphi_1=\varphi$에서 귀납하면 $\varphi_n=\varphi\circ\varphi\circ\cdots\circ\varphi=\varphi^{\circ n}$. 합성은 결합적이므로 $\varphi^{\circ(n+1)}=\varphi\circ\varphi^{\circ n}$, 즉 $\varphi_{n+1}=\varphi(\varphi_n(s))$도 같은 함수다.

**평균.** 연쇄법칙: $\varphi_{n+1}'(s)=\varphi_n'(\varphi(s))\,\varphi'(s)$. $s=1$에서 $\varphi(1)=1$이므로 $\varphi_{n+1}'(1)=\varphi_n'(1)\,m$. $\varphi_0'(1)=1$에서 $EZ_n=\varphi_n'(1)=m^n$.

**분산.** $Z_n$이 주어졌을 때 $Z_{n+1}$은 i.i.d. $Z_n$개의 합이므로 $E[Z_{n+1}\mid Z_n]=mZ_n$, $\operatorname{Var}(Z_{n+1}\mid Z_n)=Z_n\sigma^2$. 전분산 공식으로
$$\operatorname{Var}Z_{n+1}=E[Z_n\sigma^2]+\operatorname{Var}(mZ_n)=\sigma^2m^n+m^2\operatorname{Var}Z_n.$$
$v_n=\operatorname{Var}Z_n$, $v_1=\sigma^2$로 두고 풀면 $v_n=\sigma^2\sum_{k=0}^{n-1}m^{n-1+k}=\sigma^2m^{n-1}(1+m+\dots+m^{n-1})$. $m\ne1$이면 등비합 $=\sigma^2m^{n-1}\frac{m^n-1}{m-1}$, $m=1$이면 $n\sigma^2$. (귀납 확인: $\sigma^2m^n+m^2\cdot\sigma^2m^{n-1}\frac{m^n-1}{m-1}=\sigma^2m^n\frac{(m-1)+m^{n+1}-m}{m-1}=\sigma^2m^n\frac{m^{n+1}-1}{m-1}$ ✓.) $\square$

</details>

### 2.3 정리 2 — 소멸확률 정리 (Durrett 3e §1.10; Lawler 2e §2.4; G&S 4e §5.4)

> $p_1<1$이라 하자. 소멸확률 $q$는 방정식 $\varphi(s)=s$의 $[0,1]$에서 **가장 작은 해**이다. $m\le1$이면 $q=1$; $m>1$이면 $q<1$이고 $q$는 $[0,1)$의 유일한 근이다. 또한 $\varphi_n(0)\uparrow q$.

**가정이 왜 필요한가**
- $p_1<1$: $p_1=1$이면 $Z_n\equiv1$이라 $q=0$이고 $\varphi(s)=s$의 모든 점이 고정점이다 — 최소 고정점 $0=q$는 맞지만 "$m\le1\Rightarrow q=1$"이 깨진다($m=1$인데 $q=0$). $p_0=0$이면 $q=0$(소멸 불가)이고 0이 고정점이라 정리는 자명하게 성립한다.
- $p_0+p_1<1$(어떤 $k\ge2$에 $p_k>0$): $\varphi$가 강볼록이어서 고정점이 최대 둘($q$와 1). $m\le1$일 때 "1 외의 고정점 없음"을 볼록성이 준다. ($p_0+p_1=1$, $p_1<1$이면 $\varphi$가 선형이고 $m=p_1<1$, 유일 고정점 1 — 따로 처리.)
- **단조수렴**: $\{Z_n=0\}$이 증가 사건열이라 $q=\lim P(Z_n=0)=\lim\varphi_n(0)$. 이것이 "$q$가 고정점"의 출발점이다.

<details><summary>증명</summary>

**(1) $q=\lim\varphi_n(0)$.** $\{Z_n=0\}\uparrow\{\text{소멸}\}$이므로 확률의 연속성으로 $q=\lim_nP(Z_n=0)=\lim_n\varphi_n(0)$. 이 수열은 증가하고 1 이하이므로 극한이 존재한다.

**(2) $q$는 고정점.** 정리 1에서 $\varphi_{n+1}(0)=\varphi(\varphi_n(0))$. $\varphi$는 $[0,1]$에서 연속(멱급수, 수렴반경 $\ge1$)이므로 양변에 $n\to\infty$를 취하면 $q=\varphi(q)$.

**(3) 최소성.** $s^*\in[0,1]$이 임의의 고정점이라 하자. $0\le s^*$이고 $\varphi$가 증가이므로 $\varphi(0)\le\varphi(s^*)=s^*$; 귀납으로 $\varphi_n(0)\le\varphi_n(s^*)=s^*$ 모든 $n$. 극한을 취하면 $q\le s^*$.

**(4) $m\le1$이면 $q=1$.** $s<1$에서 $\varphi(s)>s$임을 보이면 1이 유일한 고정점이다. 평균값정리로 $1-\varphi(s)=\varphi(1)-\varphi(s)=\varphi'(\theta)(1-s)$, 어떤 $\theta\in(s,1)$. $\varphi$가 강볼록이면 $\varphi'$는 강증가이므로 $\varphi'(\theta)<\varphi'(1)=m\le1$, 따라서 $1-\varphi(s)<1-s$, 즉 $\varphi(s)>s$. ($\varphi$가 선형인 경우 $\varphi(s)-s=p_0(1-s)>0$.) 그러므로 $q=1$.

**(5) $m>1$이면 $q<1$.** $g(s)=\varphi(s)-s$라 하자. $g(1)=0$, $g'(1)=m-1>0$이므로 1의 바로 왼쪽에서 $g<0$. 한편 $g(0)=p_0\ge0$. $p_0>0$이면 중간값정리로 $g$의 근이 $(0,1)$에 있고, $p_0=0$이면 0이 근이다. 어느 쪽이든 $[0,1)$에 고정점 $s^*$가 있고 (3)에 의해 $q\le s^*<1$. **유일성**: $m>1$이면 어떤 $k\ge2$에 $p_k>0$이라 $g$는 강볼록; 강볼록 함수는 근을 최대 둘 가지므로($g$의 근 셋 사이에서 Rolle로 $g'$의 근 둘 — $g'$ 강증가에 모순) $[0,1]$의 근은 $q$와 1뿐이다. $\square$

</details>

### 2.4 정리 3 — 임계 과정의 생존확률 (Kolmogorov; Karlin & Taylor 2e Ch. 8)

> $m=1$, $0<\sigma^2<\infty$이면 $P(Z_n>0)=1-\varphi_n(0)\sim\dfrac{2}{\sigma^2n}$ ($n\to\infty$).

**가정이 왜 필요한가**
- $m=1$: 아임계면 $P(Z_n>0)\le EZ_n=m^n$으로 **지수적**으로, 초임계면 양의 극한 $1-q$로 — **다항식 감쇠 $1/n$은 임계에서만** 일어난다.
- $\sigma^2<\infty$: 상수 $2/\sigma^2$가 유한하려면. 분산이 무한하면 감쇠 지수가 달라진다.

증명은 인용한다(Karlin & Taylor Ch. 8; Athreya–Ney). 아이디어만: $\varphi(s)\approx s+\frac{\sigma^2}{2}(1-s)^2$ (1 근방 테일러)이므로 $u_n=1-\varphi_n(0)$은 $u_{n+1}\approx u_n-\frac{\sigma^2}{2}u_n^2$, 즉 $1/u_{n+1}-1/u_n\to\sigma^2/2$, 따라서 $u_n\sim2/(\sigma^2n)$. 노트북에서는 $1-\varphi_n(0)$을 수치로 계산해 $2/(\sigma^2n)$과 비교만 한다 — 예제 $\sigma^2=1/2$에서 $n=50$: $0.0704$ vs 근사 $0.08$. 근사는 느리게 맞아 들어간다(상대오차 $O(\log n/n)$).

### 2.5 정리 4 — $Z_n/m^n$은 마팅게일 (예고; Durrett 3e §1.10)

> $W_n=Z_n/m^n$은 $E[W_{n+1}\mid Z_0,\dots,Z_n]=W_n$을 만족하고 $EW_n=1$. $m>1$, $\sigma^2<\infty$이면 $W_n\to W$ a.s.이고 $EW=1$, $\operatorname{Var}W=\dfrac{\sigma^2}{m(m-1)}$.

**가정이 왜 필요한가**
- 마팅게일 성질에는 $E[Z_{n+1}\mid Z_n]=mZ_n$ 하나면 충분하다. **수렴**($W_n\to W$)과 극한의 분산은 05d의 $L^2$ 마팅게일 수렴 정리가 필요하다.

*스케치.* $E[Z_{n+1}\mid Z_0,\dots,Z_n]=E[Z_{n+1}\mid Z_n]=mZ_n$(마르코프 성질)을 $m^{n+1}$로 나누면 $E[W_{n+1}\mid\cdot]=Z_n/m^n=W_n$. 기댓값을 취하면 $EW_n=EW_0=1$. 분산은 정리 1에서
$$\operatorname{Var}W_n=\frac{\operatorname{Var}Z_n}{m^{2n}}=\frac{\sigma^2(1-m^{-n})}{m(m-1)}\ \uparrow\ \frac{\sigma^2}{m(m-1)}\quad(m>1),$$
유계 분산이므로 05d에서 $W_n\to W$ ($L^2$ 및 a.s.). 주 예제: $0.6875/(1.25\cdot0.25)=2.2$. 소멸 경로에서는 $W=0$이고, $P(W=0)=q$임이 알려져 있다 — 즉 살아남으면 $Z_n$은 정확히 $m^n$ 규모로 자란다.

### 2.6 직관과 함정

**직관.** 소멸은 "자손 0명"이 어느 세대에선가 실현되는 사건이다. 시조가 $k$명을 낳으면 $k$개의 독립 계보가 생기고, 전체 소멸은 그 $k$계보가 **모두** 소멸하는 것 — 각각 확률 $q$이므로 $q=\sum_kp_kq^k=\varphi(q)$. 그래서 고정점이다. 기하: $\varphi$는 볼록하고 $\varphi(1)=1$이므로 그래프는 대각선을 1에서 만나되, 1에서의 기울기 $m$이 1보다 크면 대각선 **아래**로 들어왔다가 다시 만나는 점 $q<1$이 생기고, $m\le1$이면 그래프가 대각선 위에 머물러 1이 유일한 교점이다. 코브웹 그림에서 $0\to\varphi(0)\to\varphi(\varphi(0))\to\cdots$가 계단을 오르며 $q$에 붙는 것이 세대별 소멸확률 $\varphi_n(0)$이다. 임계 $m=1$은 묘하다: 평균 개체 수는 영원히 1이지만 소멸은 확실하고, 생존확률은 $4/n$처럼 천천히 죽는다 — 살아남은 소수의 계보가 폭발적으로 커서 평균 1을 유지한다.

**함정.**
- $\varphi(s)=s$의 근은 **항상 $s=1$을 포함**한다. $q$는 '가장 작은' 비음 근이다. `np.roots`로 풀면 1이 먼저 나오기 쉽다. 0에서 $s\mapsto\varphi(s)$를 반복하면 최소 근으로 수렴하지만, 임계 $m=1$에서는 수렴이 $2/(\sigma^2n)$ 속도라 $10^4$번 반복해도 $1-2/(\sigma^2\cdot10^4)\approx0.9996$에서 멈춘다(4.1절에서 직접 확인). 그래서 `extinction_probability`는 $m\le1$이면 반복 대신 정리 2를 그대로 써서 정확히 1을 돌려주고, $m>1$일 때만 반복한다.
- $EZ_n=m^n\to\infty$여도 소멸할 수 있다(주 예제: $m=1.25$인데 $q=1/2$). 평균은 살아남은 경로의 폭발이 끌어올린 것.
- 시뮬레이션의 "세대 $n$까지 소멸 비율"은 $\varphi_n(0)$이지 $q$가 아니다. $n$이 작으면 $q$보다 눈에 띄게 작다($\varphi_{10}(0)=0.489$ vs $q=0.5$). 비교 기준을 $\varphi_n(0)$으로 둘 것.
- $Z_n$의 분산은 $m^{2n}$ 규모라 $EZ_n$의 MC 추정 SE가 크다. 20000경로에서 $n=10$의 상대 SE $\approx1\%$; $n=25$에서는 평균 추정을 신뢰하기 어렵다(무거운 꼬리, 상대 SE $\approx1\%$지만 극소수 경로가 지배).
- 무한 지지 자손분포(푸아송)를 절단해서 시뮬레이션하면 pgf도 **같은 절단 pmf**로 계산해야 정확히 비교된다(절단 40에서 손실 질량 $<10^{-16}$이라 실질 영향은 없다).

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — (1) $q=\varphi(q)$가 성립하는 확률적 이유, (2) $m\le1$이면 $q=1$인 기하학적 이유, (3) 임계 과정에서 '평균 1, 소멸 확률 1'이 모순이 아닌 이유를 각각 두 문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

주 예제는 자손분포 $p=(p_0,p_1,p_2)=(0.25,0.25,0.5)$ on $\{0,1,2\}$이다. 숫자를 먼저 적어 보세요 — 틀려도 됩니다. 정확값은 5절에서 대조합니다. (힌트: $\varphi(s)=\frac14+\frac14s+\frac12s^2$는 2차식이라 고정점을 손으로 풀 수 있다.)

In [ ]:
predictions = {
    "m_main": None,              # 주 예제 (0.25, 0.25, 0.5)의 평균 자손 수 m 은?
    "q_main": None,              # 주 예제의 소멸확률 q (φ(s)=s 의 최소 근) 는?
    "ext_by_10": None,           # 10세대 안에 소멸할 확률 φ_10(0) 은? (q 보다 작다 — 얼마나?)
    "ext_by_25": None,           # 25세대 안에 소멸할 확률 φ_25(0) 은?
    "mean_z10": None,            # E Z_10 = m^10 은?
    "sd_z10": None,              # Z_10 의 표준편차 sqrt(Var Z_10) 은? (σ² = Eξ² - m² = 2.25 - 1.5625)
    "q_poisson15": None,         # 자손 ~ Poisson(1.5) 일 때 소멸확률 q = e^{1.5(q-1)} 의 최소 근은?
    "ext_poisson_by_25": None,   # Poisson(1.5) 과정이 25세대 안에 소멸할 확률 φ_25(0) 은?
    "surv_critical_50": None,    # 임계 (0.25, 0.5, 0.25) (m=1, σ²=0.5) 의 50세대 생존확률 1-φ_50(0) 은? (Kolmogorov 근사 2/(σ² n) 과 비교)
    "mean_z50_critical": None,   # 임계 과정의 E Z_50 은?
}

## 4. Simulate — 시뮬레이션

### 4.0 도우미 함수

`spkit`에 없는 네 개를 여기서 정의한다: pgf 평가, 0에서의 반복 $\varphi_k(0)$, 분산 공식, 코브웹 그리기. 소멸확률 자체는 `spkit.markov.extinction_probability`를 그대로 쓴다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def pgf(s, pmf):
    """φ(s) = Σ p_k s^k  (Horner; s 는 스칼라 또는 배열)."""
    s = np.asarray(s, dtype=float)
    out = np.zeros_like(s)
    for p in reversed(list(pmf)):
        out = out * s + p
    return out if out.ndim else float(out)

def pgf_iterates(pmf, n):
    """[φ_0(0)=0, φ_1(0), ..., φ_n(0)] — 세대 n 까지의 소멸확률."""
    s, out = 0.0, [0.0]
    for _ in range(n):
        s = pgf(s, pmf)
        out.append(s)
    return np.array(out)

def gw_variance(m, sigma2, n):
    """Var Z_n = σ² m^{n-1} (m^n - 1)/(m - 1)  (m ≠ 1),  n σ²  (m = 1)."""
    return sigma2 * m ** (n - 1) * (m ** n - 1) / (m - 1) if m != 1 else n * sigma2

def cobweb(ax, pmf, n_steps, **kw):
    """0 → φ(0) → φ(φ(0)) → ... 계단을 그린다."""
    s, xs, ys = 0.0, [0.0], [0.0]
    for _ in range(n_steps):
        t = pgf(s, pmf)
        xs += [s, t]; ys += [t, t]; s = t
    ax.plot(xs, ys, **kw)

### 4.1 pgf와 코브웹 — 소멸확률은 최소 고정점

주 예제 $\varphi(s)=\frac14+\frac14s+\frac12s^2$의 고정점은 $2s^2-3s+1=0$, 즉 $s\in\{1/2,1\}$. 0에서 반복하면 작은 쪽 $q=1/2$로 간다: 오차는 $q-\varphi_n(0)\approx C\cdot\varphi'(q)^n=C\cdot0.75^n$으로 줄어든다. 같은 축에 임계 $(0.25,0.5,0.25)$와 아임계 $(0.5,0.3,0.2)$ ($m=0.7$)의 $\varphi$도 겹친다 — 둘 다 대각선 위에 머물러 1 외의 교점이 없다.

In [ ]:
pmf_main = np.array([0.25, 0.25, 0.5])
pmf_crit = np.array([0.25, 0.5, 0.25])
pmf_sub = np.array([0.5, 0.3, 0.2])
ks = np.arange(3)
m_main, sigma2_main = pmf_main @ ks, pmf_main @ ks**2 - (pmf_main @ ks) ** 2
q_main = extinction_probability(pmf_main)
roots = np.roots([0.5, 0.25 - 1, 0.25])            # 0.5 s² + (0.25-1) s + 0.25 = 0 의 근 (1 이 먼저 나온다: 함정 1)
phi_main = pgf_iterates(pmf_main, 30)
print(f"m = {m_main}, sigma^2 = {sigma2_main}, np.roots -> {np.sort(roots)}, extinction_probability -> {q_main:.10f}")
print("phi_n(0), n = 1..10:", np.round(phi_main[1:11], 5))
print(f"phi_10(0) = {phi_main[10]:.6f}, phi_25(0) = {phi_main[25]:.6f}, phi_30(0) = {phi_main[30]:.7f}")
print("error ratio (q - phi_{n+1}(0)) / (q - phi_n(0)) at n=25:", (q_main - phi_main[26]) / (q_main - phi_main[25]))
print("m of critical / subcritical pmf:", pmf_crit @ ks, pmf_sub @ ks)
assert abs(q_main - 0.5) < 1e-8
s_naive = 0.0                                          # 함정 1: 임계 pmf 를 0 에서 10^4 번 순진하게 반복하면?
for _ in range(10_000):
    s_naive = pgf(s_naive, pmf_crit)
print(f"critical pmf: naive iteration x 10^4 -> {s_naive:.6f} (~ 1 - 2/(sigma^2 n) = {1 - 2/(0.5*10_000):.4f}),"
      f"  extinction_probability -> {extinction_probability(pmf_crit)} (theorem 2: m <= 1 => q = 1)")

In [ ]:
s = np.linspace(0, 1, 201)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(s, pgf(s, pmf_main), "C0", lw=2, label="phi(s), main (0.25, 0.25, 0.5), m = 1.25")
ax.plot(s, pgf(s, pmf_crit), "C1", lw=1, label="critical (0.25, 0.5, 0.25), m = 1")
ax.plot(s, pgf(s, pmf_sub), "C2", lw=1, label="subcritical (0.5, 0.3, 0.2), m = 0.7")
ax.plot(s, s, "k--", lw=1, label="diagonal s")
cobweb(ax, pmf_main, 30, color="C3", lw=0.9, label="cobweb 0 -> phi(0) -> phi(phi(0)) -> ...")
ax.plot([q_main], [q_main], "C3o", ms=8, label=f"q = {q_main:.3f} (smallest fixed point)")
ax.set(xlabel="s", ylabel="phi(s)", xlim=(0, 1), ylim=(0, 1), aspect="equal",
       title="Extinction probability = smallest fixed point of the offspring pgf")
ax.legend(fontsize=8, loc="upper left"); plt.show()

계단은 $0.25,\,0.344,\,0.395,\dots$로 오르며 $\varphi_{10}(0)=0.489$, $\varphi_{25}(0)=0.49985$가 된다. 임계·아임계 곡선은 대각선에 1에서만 접하거나 위에 있다 — 반복하면 1로 가지만, 임계는 $1/n$ 속도로 **매우 느리게** 붙는다(4.5절): 위 출력처럼 $10^4$번 반복해도 $0.9996$에서 멈추므로 `extinction_probability`는 $m\le1$이면 정리 2로 바로 1을 돌려준다(함정 1).

### 4.2 주 예제 시뮬레이션 — 소멸 비율은 $\varphi_n(0)$을 따라간다

`galton_watson`은 세대마다 `rng.multinomial(Z, pmf)`로 "자손 0명·1명·2명을 낳은 개체 수"를 뽑아 더하므로 개체 수가 커져도 비용이 늘지 않는다. 경로 20개(symlog 축: 0은 0, 그 위는 로그)와 세대별 소멸 비율을 그린다.

In [ ]:
N_GEN = 25
Z = galton_watson(pmf_main, N_GEN, rng, N_PATHS)
gens = np.arange(N_GEN + 1)
ext_frac = (Z == 0).mean(axis=0)                     # 세대별 소멸 비율 (MC)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
plot_paths(gens, Z, ax=ax1, n_show=20, alpha=0.6)
ax1.set(yscale="symlog", xlabel="generation n", ylabel="Z_n (symlog)", title="20 sample paths (extinct paths end at 0)")
ax2.plot(gens, phi_main[:N_GEN + 1], "C0-", label="phi_n(0) exact")
ax2.plot(gens, ext_frac, "C3o", ms=4, label=f"MC fraction extinct by n ({N_PATHS} paths)")
ax2.axhline(q_main, color="k", ls="--", lw=1, label="q = 1/2")
ax2.set(xlabel="generation n", ylabel="P(Z_n = 0)", ylim=(0, 0.6), title="Galton-Watson (0.25, 0.25, 0.5): extinction fraction -> q = 1/2")
ax2.legend(loc="lower right"); plt.show()

In [ ]:
phi10_0, phi25_0 = phi_main[10], phi_main[25]
est_ext10 = mc_proportion(Z[:, 10] == 0)
est_ext25 = mc_proportion(Z[:, 25] == 0)
est_mean10 = mc_estimate(Z[:, 10])
mean_z10, var_z10 = m_main ** 10, gw_variance(m_main, sigma2_main, 10)
print(f"extinct by 10: {est_ext10}   exact phi_10(0) = {phi10_0:.6f}")
print(f"extinct by 25: {est_ext25}   exact phi_25(0) = {phi25_0:.6f}  (q = {q_main:.4f})")
print(f"mean Z_10    : {est_mean10}   exact m^10 = {mean_z10:.6f}, sd = {np.sqrt(var_z10):.3f}")
print(f"sample var Z_5 = {Z[:, 5].var(ddof=1):.3f}   exact Var Z_5 = {gw_variance(m_main, sigma2_main, 5):.4f} (E2)")
print(f"max Z_25 = {Z[:, 25].max()},  median Z_25 = {np.median(Z[:, 25]):.0f},  mean Z_25 = {Z[:, 25].mean():.1f} (exact {m_main**25:.1f})")

세대 10에서 이미 절반 가까이($0.489$) 소멸했고 25세대엔 $q$와 $10^{-4}$ 차이다. 생존 경로는 $m^n$ 규모로 자라 $Z_{25}$의 중앙값은 0인데 평균은 $1.25^{25}\approx265$ — 함정 2.

### 4.3 마팅게일 예고 — $W_n=Z_n/m^n$은 안정된다

In [ ]:
W = Z / m_main ** gens
fig, ax = plt.subplots()
plot_paths(gens, W, ax=ax, n_show=20, alpha=0.6)
ax.axhline(1, color="k", ls="--", lw=1, label="E W_n = 1")
ax.set(xlabel="generation n", ylabel="W_n = Z_n / 1.25^n", title="Z_n / m^n stabilizes: a martingale (05d)")
ax.legend(); plt.show()
var_w25 = sigma2_main * (1 - m_main ** -25) / (m_main * (m_main - 1))
print(f"mean W_25 = {mc_estimate(W[:, 25])}   (exact E W_25 = 1; Var W_25 = {var_w25:.4f}, limit {sigma2_main/(m_main*(m_main-1)):.2f})")
print(f"fraction of paths with W_25 == 0: {(W[:, 25] == 0).mean():.4f}  (-> q = 0.5)")

소멸 경로는 0에 눕고, 생존 경로는 각자 다른 양의 값으로 **안정**된다 — 랜덤 극한 $W$. 표본평균은 1 근처지만 $\operatorname{sd}(W_{25})\approx\sqrt{2.2}$라 SE가 크다. 05d에서 $W_n\to W$를 증명한다.

### 4.4 푸아송 자손 — $q(\lambda)$의 상전이

자손 $\sim\text{Poisson}(\lambda)$이면 $\varphi(s)=e^{\lambda(s-1)}$이고 $q$는 $q=e^{\lambda(q-1)}$의 최소 근이다. 시뮬레이션은 지지를 $0..40$으로 절단한 pmf를 쓰므로 pgf도 같은 절단 pmf로 계산한다(함정 5). `extinction_probability`(반복)와 `brentq`(연속 방정식의 근)가 $10^{-8}$ 안에서 같아야 한다.

In [ ]:
LAM = 1.5
pmf_p = poisson.pmf(np.arange(41), LAM)
print(f"truncation at 40: lost mass = {1 - pmf_p.sum():.2e}")
q_pois_iter = extinction_probability(pmf_p)
q_pois_root = brentq(lambda q: q - np.exp(LAM * (q - 1)), 1e-9, 1 - 1e-9)
phi_pois = pgf_iterates(pmf_p, 25)
phi25_poisson = phi_pois[25]
print(f"q(1.5): iteration {q_pois_iter:.10f}, brentq {q_pois_root:.10f};  phi_10(0) = {phi_pois[10]:.6f}, phi_25(0) = {phi25_poisson:.7f}")
assert abs(q_pois_iter - q_pois_root) < 1e-8
Zp = galton_watson(pmf_p, 25, rng, N_PATHS)
est_ext25_p = mc_proportion(Zp[:, 25] == 0)
print(f"extinct by 25: {est_ext25_p}   exact phi_25(0) = {phi25_poisson:.6f};  max Z_25 = {Zp[:, 25].max():.3g}")

In [ ]:
lam_grid = np.linspace(0.5, 3, 51)
q_curve = np.array([extinction_probability(poisson.pmf(np.arange(41), lam)) for lam in lam_grid])
phi25_curve = np.array([pgf_iterates(poisson.pmf(np.arange(41), lam), 25)[25] for lam in lam_grid])
lam_mc = [0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
mc_pts = [mc_proportion(galton_watson(poisson.pmf(np.arange(41), lam), 25, rng, N_CURVE)[:, 25] == 0) for lam in lam_mc]
fig, ax = plt.subplots()
ax.plot(lam_grid, q_curve, "C0-", lw=2, label="q(lambda) exact (smallest fixed point)")
ax.plot(lam_grid, phi25_curve, "C0:", lw=1, label="phi_25(0): extinct within 25 generations")
ax.errorbar(lam_mc, [r.mean for r in mc_pts], yerr=[2 * r.se for r in mc_pts], fmt="C3o", capsize=3, label=f"MC extinct by 25 ({N_CURVE} paths, +/-2 SE)")
ax.axvline(1, color="k", ls="--", lw=1, label="m = 1 (critical)")
ax.set(xlabel="lambda = m", ylabel="extinction probability", ylim=(0, 1.05),
       title="Phase transition at m = 1: extinction probability of Poisson(lambda) branching")
ax.legend(fontsize=8); plt.show()
for lam, r in zip(lam_mc, mc_pts):
    print(f"lambda = {lam}: q = {extinction_probability(poisson.pmf(np.arange(41), lam)):.5f}, MC extinct-by-25 = {r}")

$\lambda\le1$에서 $q=1$, $\lambda>1$에서 급격히 떨어진다 — 임계점에서 미분이 불연속인 **상전이**. MC 점은 $q$가 아니라 $\varphi_{25}(0)$(점선)을 따른다: $\lambda=1$에서 25세대 안 소멸은 $\varphi_{25}(0)\approx0.93$뿐이다(함정 3; Kolmogorov 근사 $1-2/25=0.92$는 이보다 약간 낮다). 25세대 안에 $3^{25}$ 규모까지 자라도 `multinomial` 덕에 비용은 같다.

### 4.5 임계 과정 — 평균 1, 소멸 확률 1

$(0.25,0.5,0.25)$: $\varphi(s)=\frac14(1+s)^2$, $m=1$, $\sigma^2=0.5$. $EZ_n=1$이 영원히 유지되지만 생존확률 $1-\varphi_n(0)$은 $2/(\sigma^2n)=4/n$처럼 죽는다.

In [ ]:
m_crit, sigma2_crit = pmf_crit @ ks, pmf_crit @ ks**2 - (pmf_crit @ ks) ** 2
phi_crit = pgf_iterates(pmf_crit, 50)
phi50_crit = phi_crit[50]
mean_z50_crit = m_crit ** 50                          # = 1 (정리 1, m = 1)
Zc = galton_watson(pmf_crit, 50, rng, N_PATHS)
est_surv50 = mc_proportion(Zc[:, 50] > 0)
est_mean50 = mc_estimate(Zc[:, 50])
print(f"survival by 50: {est_surv50}   exact 1 - phi_50(0) = {1 - phi50_crit:.6f},  Kolmogorov 2/(sigma^2 n) = {2/(sigma2_crit*50):.4f}")
print(f"mean Z_50     : {est_mean50}   exact m^50 = {mean_z50_crit:.1f},  sd = sqrt(n sigma^2) = {np.sqrt(50*sigma2_crit):.2f}")
print(f"Z_50 among survivors: mean {Zc[Zc[:, 50] > 0, 50].mean():.1f}  (exact 1/(1-phi_50(0)) = {1/(1-phi50_crit):.1f})")
n_arr = np.arange(1, 51)
fig, ax = plt.subplots()
ax.loglog(n_arr, 1 - phi_crit[1:], "C0-", lw=2, label="P(Z_n > 0) = 1 - phi_n(0) exact")
n_mark = np.array([1, 2, 5, 10, 20, 50])
ax.loglog(n_mark, (Zc[:, n_mark] > 0).mean(axis=0), "C3o", label=f"MC survival fraction ({N_PATHS} paths)")
ax.loglog(n_arr, 2 / (sigma2_crit * n_arr), "k--", lw=1, label="Kolmogorov 2/(sigma^2 n) = 4/n")
ax.set(xlabel="generation n", ylabel="survival probability", title="Critical case: mean stays 1, survival decays like 2/(sigma^2 n)")
ax.legend(); plt.show()

로그-로그에서 기울기 $-1$: 지수 감쇠(아임계)도 양의 극한(초임계)도 아닌 **다항식** 감쇠. 근사 $4/n$은 위에서 천천히 붙는다($n=50$: $0.080$ vs $0.070$). 생존자만 보면 $E[Z_{50}\mid Z_{50}>0]=1/(1-\varphi_{50}(0))\approx14$ — 소수의 큰 계보가 평균 1을 지탱한다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

먼저 MC 추정이 없는 네 항목(닫힌 형식만 있는 $m$, $q$, $\operatorname{sd}Z_{10}$, 푸아송 $q$)을 예측과 나란히 놓고, 나머지 여섯은 아래 표에서 예측·MC·정확값을 대조한다.

In [ ]:
for key, exact in [("m_main", m_main), ("q_main", q_main), ("sd_z10", np.sqrt(var_z10)), ("q_poisson15", q_pois_root)]:
    print(f"{key:12s} predicted = {str(predictions[key]):>8}   exact = {exact:.6f}")

In [ ]:
rows = [
    {"name": "ext_by_10 (phi_10(0))", "predicted": predictions["ext_by_10"],
     "estimate": est_ext10, "exact": phi10_0},
    {"name": "ext_by_25 (phi_25(0))", "predicted": predictions["ext_by_25"],
     "estimate": est_ext25, "exact": phi25_0},
    {"name": "mean_z10 (m^10)", "predicted": predictions["mean_z10"],
     "estimate": est_mean10, "exact": mean_z10},
    {"name": "ext_poisson_by_25", "predicted": predictions["ext_poisson_by_25"],
     "estimate": est_ext25_p, "exact": phi25_poisson},
    {"name": "surv_critical_50 (1 - phi_50(0))", "predicted": predictions["surv_critical_50"],
     "estimate": est_surv50, "exact": 1 - phi50_crit},
    {"name": "mean_z50_critical", "predicted": predictions["mean_z50_critical"],
     "estimate": est_mean50, "exact": mean_z50_crit},
]
Markdown(compare_table(rows))

- **ext_by_10 / ext_by_25**: 어긋나면 `galton_watson`의 자손 추출이 pmf와 다르거나, 비교 기준을 $q$로 잘못 잡은 것이다($\varphi_{10}(0)=0.489\ne0.5$: 함정 3).
- **mean_z10**: 어긋나면 $EZ_n=m^n$(정리 1)이나 시뮬레이션의 독립성이 깨진 것. SE $\approx13/\sqrt{N}$으로 꼬리가 무거워 검사가 느슨하다.
- **ext_poisson_by_25**: 어긋나면 절단 pmf와 pgf가 서로 다른 분포를 쓰고 있다(함정 5).
- **surv_critical_50**: 어긋나면 임계 과정의 소멸이 $\varphi_n(0)$을 따르지 않는다는 뜻 — 정리 2의 (1)–(2)가 깨진 것. 값이 0.08이면 Kolmogorov 근사를 정확값으로 착각한 것이다.
- **mean_z50_critical**: 어긋나면 마팅게일 성질 $EZ_n=1$ 자체가 깨진 것. SE $\approx4.9/\sqrt N$.

예측값 `predictions`는 표에만 나타나며, 아래 검정은 **MC 추정 vs 닫힌 형식**만 본다. 예측 셀에 적은 숫자와 표의 정확값이 다르면 그 이유를 2절에서 찾아 보세요.

In [ ]:
# 손으로 푼 값과 교차검증 (정확값끼리)
assert abs(extinction_probability([.25, .25, .5]) - 0.5) < 1e-8
assert abs(extinction_probability(pmf_p) - brentq(lambda q: q - np.exp(1.5 * (q - 1)), 1e-9, 1 - 1e-9)) < 1e-8
assert np.isclose(pgf_iterates([.25, .25, .5], 10)[10], 0.4888206932) and np.isclose(pgf_iterates([.25, .25, .5], 25)[25], 0.4998548922)
# MC vs 닫힌 형식 (k = 4)
assert_close(est_ext10, phi10_0, k=4, name="ext_by_10")
assert_close(est_ext25, phi25_0, k=4, name="ext_by_25")
assert_close(est_mean10, mean_z10, k=4, name="mean_z10")
assert_close(est_ext25_p, phi25_poisson, k=4, name="ext_poisson_by_25")
assert_close(est_surv50, 1 - phi50_crit, k=4, name="surv_critical_50")
assert_close(est_mean50, mean_z50_crit, k=4, name="mean_z50_critical")
print("all MC-vs-exact checks passed (k=4)")

## 6. 연습문제

### E1 계산 — 이진 분열 (binary fission)

각 개체가 확률 $p$로 자손 2, 확률 $1-p$로 자손 0을 남긴다. $m$, $\varphi(s)$, 소멸확률 $q$를 $p$의 함수로 구하고 $p=0.6$에서 값을 내시오. $q<1$인 $p$의 범위는? (`extinction_probability([1-p, 0, p])`로 확인.)

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$m=2p$, $\varphi(s)=(1-p)+ps^2$. 고정점 방정식 $ps^2-s+(1-p)=0$의 근은
$$s=\frac{1\pm\sqrt{1-4p(1-p)}}{2p}=\frac{1\pm|2p-1|}{2p}\ \Rightarrow\ s\in\Big\{1,\ \frac{1-p}{p}\Big\}.$$
$p>1/2$ ($m>1$)이면 $(1-p)/p<1$이 최소 근이라 $q=(1-p)/p$; $p\le1/2$ ($m\le1$)이면 $(1-p)/p\ge1$이라 $q=1$ (정리 2). 따라서 $q<1\iff p>1/2\iff m>1$. $p=0.6$: $m=1.2$, $q=0.4/0.6=2/3\approx0.6667$. 확률적 해석: $q=\varphi(q)=0.4+0.6q^2$ — 시조가 죽거나($0.4$), 둘을 낳고 두 계보가 모두 소멸($0.6q^2$).

```python
p = 0.6
print(extinction_probability([1 - p, 0, p]), (1 - p) / p, "m =", 2 * p)      # 0.6667 0.6667 m = 1.2
for p in [0.3, 0.5, 0.55, 0.8]:
    print(p, round(extinction_probability([1 - p, 0, p]), 6), round(min(1, (1 - p) / p), 6))
```

</details>

### E2 유도 후 시뮬레이션 검증 — $\operatorname{Var}Z_n$

점화식 $\operatorname{Var}Z_{n+1}=\sigma^2m^n+m^2\operatorname{Var}Z_n$을 (전분산 공식으로) 유도하고 $\operatorname{Var}Z_n=\sigma^2m^{n-1}(m^n-1)/(m-1)$ ($m\ne1$)을 얻으시오. 주 예제에서 $\operatorname{Var}Z_5$를 계산하고 `N_PATHS`경로의 표본분산 `Z[:, 5].var(ddof=1)`과 비교하시오 — 표본분산의 SE는 $\operatorname{sd}\big((Z_5-\bar Z_5)^2\big)/\sqrt n$으로 근사하고 `assert_close(..., se=...)`로 검정하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** $Z_n$이 주어지면 $Z_{n+1}$은 i.i.d. $Z_n$개의 합이므로(00c 랜덤 합) $E[Z_{n+1}\mid Z_n]=mZ_n$, $\operatorname{Var}(Z_{n+1}\mid Z_n)=Z_n\sigma^2$. 전분산 공식
$$\operatorname{Var}Z_{n+1}=E[\operatorname{Var}(Z_{n+1}\mid Z_n)]+\operatorname{Var}(E[Z_{n+1}\mid Z_n])=\sigma^2EZ_n+m^2\operatorname{Var}Z_n=\sigma^2m^n+m^2\operatorname{Var}Z_n.$$
$v_n=\operatorname{Var}Z_n$, $v_1=\sigma^2$. $v_n/m^{2n}$으로 나누면 $\frac{v_{n+1}}{m^{2n+2}}=\frac{v_n}{m^{2n}}+\frac{\sigma^2}{m^{n+2}}$ — 망원합으로 $v_n=\sigma^2\sum_{k=0}^{n-1}m^{n-1+k}=\sigma^2m^{n-1}\frac{m^n-1}{m-1}$.

**주 예제 $n=5$.** $\sigma^2=0.6875$, $m=1.25$: $\operatorname{Var}Z_5=0.6875\cdot1.25^4\cdot(1.25^5-1)/0.25=13.7752$ (sd $3.71$), $EZ_5=1.25^5=3.0518$. 20000경로의 표본분산은 $\approx13.7$, SE $\approx0.17$ — 4 SE 안에서 일치한다. 표본분산의 SE가 $\operatorname{sd}((Z-\bar Z)^2)/\sqrt n$인 이유: 표본분산은 $(Z_i-\bar Z)^2$의 평균이므로 그 평균의 SE.

```python
var5_exact = gw_variance(m_main, sigma2_main, 5)
z5 = Z[:, 5].astype(float)
var5_hat = z5.var(ddof=1)
se_var5 = ((z5 - z5.mean()) ** 2).std(ddof=1) / np.sqrt(z5.size)
print(f"Var Z_5: sample {var5_hat:.3f} (SE {se_var5:.3f}) vs exact {var5_exact:.4f};  E Z_5 = {m_main**5:.4f}, sample {z5.mean():.4f}")
assert_close(var5_hat, var5_exact, se=se_var5, k=4, name="var_z5")
```

</details>

### E3 가정을 깨보기 — 임계 근방에서 "$m>1$이면 산다"는 직관

자손 pmf $(0.26,0.5,0.24)$ ($m=0.98$), $(0.25,0.5,0.25)$ ($m=1$), $(0.24,0.5,0.26)$ ($m=1.02$)에 대해 $q$와 50세대 생존확률 $1-\varphi_{50}(0)$을 계산하고 `N_PATHS`경로로 확인하시오(`mc_proportion(Z[:,50] > 0)` + `assert_close`). 세 경우 $EZ_{50}=m^{50}$은 각각 얼마이며, 그것이 '전형적인 경로'(중앙값)를 얼마나 잘못 묘사하는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$q=1,\ 1,\ 0.92308$ (마지막은 $\varphi(s)=0.24+0.5s+0.26s^2=s$ ⇒ $0.26s^2-0.5s+0.24=0$ ⇒ $s=0.24/0.26=12/13$). 50세대 생존확률 $0.03935,\ 0.07042,\ 0.11325$ — $m$이 2%씩 다를 뿐인데 궁극 운명은 "확실 소멸 / 확실 소멸 / 7.7% 생존"으로 갈리고, 50세대에서는 그 차이가 아직 작다. $EZ_{50}=0.98^{50}=0.364,\ 1,\ 1.02^{50}=2.69$: 세 경우 모두 **약 89–96%의 경로가 이미 0**(중앙값 0; $m=1.02$에서도 $1-0.113=88.7\%$)인데 평균은 소수 생존 경로가 지탱한다 — 평균은 전형을 말해 주지 않는다. `extinction_probability`는 $m\le1$인 앞의 두 경우에 정리 2로 정확히 1을 돌려준다(함정 1).

```python
for pm in [(0.26, 0.5, 0.24), (0.25, 0.5, 0.25), (0.24, 0.5, 0.26)]:
    pm = np.array(pm); m = pm @ ks
    surv50 = 1 - pgf_iterates(pm, 50)[50]
    Z3 = galton_watson(pm, 50, rng, N_PATHS)
    est = mc_proportion(Z3[:, 50] > 0)
    assert_close(est, surv50, k=4, name=f"surv50 m={m:.2f}")
    print(f"m = {m:.2f}: q = {extinction_probability(pm):.5f}, "
          f"survival@50 exact {surv50:.5f} vs MC {est}, E Z_50 = {m**50:.3f}, MC mean {Z3[:, 50].mean():.3f}, median {np.median(Z3[:, 50]):.0f}")
```

</details>

## 7. 정리

1. GW 과정은 $\mathbb N$ 위의 마르코프 사슬(0 흡수, $p_0>0$이면 나머지 일시); $\varphi_n=\varphi^{\circ n}$, $EZ_n=m^n$, $\operatorname{Var}Z_n=\sigma^2m^{n-1}(m^n-1)/(m-1)$ (정리 1, 랜덤 합의 반복).
2. 소멸확률 $q$ = $\varphi(s)=s$의 **최소** 비음 근 = $\lim\varphi_n(0)$; $m\le1\Rightarrow q=1$, $m>1\Rightarrow q<1$ — 볼록성과 1에서의 기울기 $m$이 결정한다(정리 2, 코브웹).
3. 주 예제 $(0.25,0.25,0.5)$: $m=1.25$, $q=1/2$, $\varphi_{10}(0)=0.489$, $\varphi_{25}(0)=0.49985$; 푸아송(1.5): $q=0.4172$. 시뮬레이션의 세대별 소멸 비율은 $\varphi_n(0)$과 4 SE 안에서 일치했다.
4. 임계 $m=1$: 평균 1 유지, 소멸 확률 1, 생존확률 $\sim2/(\sigma^2n)$ (Kolmogorov, 로그-로그 기울기 $-1$); 임계 근방 2%의 $m$ 차이가 운명을 가른다(E3).
5. $Z_n/m^n$은 마팅게일(정리 4; 05d에서 수렴 증명), 극한 분산 $\sigma^2/(m(m-1))=2.2$; 소멸 경로는 0, 생존 경로는 $m^n$ 규모.

**trap 카드.** Q: $EZ_n=m^n\to\infty$이면 개체군은 확률 1로 살아남는가? 반대로 $m=1$이면 평균이 일정하니 살아남는가?
→ A: 둘 다 아니다. 초임계 예제는 $EZ_n\to\infty$지만 절반이 소멸($q=1/2$); 임계는 평균 1인데 확률 1로 소멸한다. 평균은 소수의 폭발적 생존 경로가 끌어올릴 뿐 전형적 경로를 말하지 않는다 — 소멸확률은 pgf 고정점으로 따로 계산해야 한다.

**다음 노트북: 01x (Pólya의 랜덤 워크 재귀성).** 무한 상태 사슬의 두 번째 예: 대칭 랜덤 워크는 기약이지만 차원에 따라 재귀(1, 2차원)와 일시(3차원 이상)가 갈린다. 유한 사슬 분류 정리가 왜 '유한'을 요구했는지 답한다. 선택 노트북이므로 건너뛰면 01r(복습)로.

`log/progress.md` 한 줄 템플릿:
`- YYYY-MM-DD 01g 분기 과정 — 예측 적중 _/10, compare 판정 ok _/6, E1 _ E2 _ E3 _, 막힌 곳: ___`